# Lesson 25: Imbalanced Data & Anomaly Detection

*ML Academy — companion notebook*

Rare events: the right metrics, thresholds vs class weights vs resampling, SMOTE, and finding anomalies without labels with z-scores, Isolation Forest and LOF.

Run each cell in order with **Shift+Enter**. The explanations, maths and worked examples are in `lesson-25.html`; this notebook contains every code example from the lesson so you can run and modify it.

Requirements: `pip install numpy pandas scikit-learn matplotlib`

## 2. Why accuracy fails, in numbers

In [1]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, recall_score, precision_score, average_precision_score

# 1% positives, like fraud
X, y = make_classification(n_samples=20000, n_features=10, n_informative=5, weights=[0.99],
                           class_sep=1.0, flip_y=0, random_state=0)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.5, random_state=0, stratify=y)
print("positives in the test set:", int(y_te.sum()), "of", len(y_te))
for name, m in [("always 'normal'", DummyClassifier(strategy="most_frequent")), ("logistic regression", LogisticRegression(max_iter=1000))]:
    m.fit(X_tr, y_tr); pr = m.predict(X_te)
    print(f"{name:20}: accuracy {accuracy_score(y_te, pr):.3f}   recall {recall_score(y_te, pr):.3f}")

positives in the test set: 100 of 10000
always 'normal'     : accuracy 0.990   recall 0.000
logistic regression : accuracy 0.995   recall 0.520


## 3. Thresholds, class weights and resampling

In [2]:
def report(name, scores, threshold=0.5):
    flag = scores >= threshold
    print(f"{name:32}: recall {recall_score(y_te, flag):.3f}  precision {precision_score(y_te, flag, zero_division=0):.3f}  "
          f"average precision {average_precision_score(y_te, scores):.3f}")

plain = LogisticRegression(max_iter=1000).fit(X_tr, y_tr).predict_proba(X_te)[:, 1]
weighted = LogisticRegression(max_iter=1000, class_weight="balanced").fit(X_tr, y_tr).predict_proba(X_te)[:, 1]
rng = np.random.default_rng(0)
pos = np.flatnonzero(y_tr == 1)
extra = rng.choice(pos, size=(y_tr == 0).sum() - len(pos), replace=True)        # random oversampling
Xo, yo = np.vstack([X_tr, X_tr[extra]]), np.r_[y_tr, y_tr[extra]]
over = LogisticRegression(max_iter=1000).fit(Xo, yo).predict_proba(X_te)[:, 1]

report("plain, threshold 0.5", plain)
report("plain, threshold 0.1", plain, 0.1)
report("class_weight='balanced'", weighted)
report("random oversampling", over)

plain, threshold 0.5            : recall 0.520  precision 0.912  average precision 0.746
plain, threshold 0.1            : recall 0.760  precision 0.490  average precision 0.746
class_weight='balanced'         : recall 0.900  precision 0.146  average precision 0.471
random oversampling             : recall 0.900  precision 0.146  average precision 0.470


## 4. SMOTE: synthetic minority examples

In [3]:
# SMOTE from scratch: new minority points on the line between a point and one of its minority neighbours
from sklearn.neighbors import NearestNeighbors

def smote(Xmin, n_new, k=5, seed=0):
    rng = np.random.default_rng(seed)
    nn = NearestNeighbors(n_neighbors=k + 1).fit(Xmin)
    _, nbrs = nn.kneighbors(Xmin)
    i = rng.integers(0, len(Xmin), n_new)
    j = nbrs[i, rng.integers(1, k + 1, n_new)]          # skip column 0 (the point itself)
    gap = rng.random((n_new, 1))
    return Xmin[i] + gap * (Xmin[j] - Xmin[i])

Xmin = X_tr[y_tr == 1]
new = smote(Xmin, (y_tr == 0).sum() - len(Xmin))
Xs, ys = np.vstack([X_tr, new]), np.r_[y_tr, np.ones(len(new), int)]
report("SMOTE", LogisticRegression(max_iter=1000).fit(Xs, ys).predict_proba(X_te)[:, 1])
print("synthetic minority points created:", len(new))

SMOTE                           : recall 0.900  precision 0.169  average precision 0.515
synthetic minority points created: 9800


## 5. Anomaly detection without labels

In [4]:
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import roc_auc_score as roc_auc

# Unsupervised anomaly detection: 980 normal points in two clusters, 20 anomalies scattered around them
rng = np.random.default_rng(1)
normal = np.vstack([rng.normal([0, 0], 0.6, (490, 2)), rng.normal([4, 4], 0.6, (490, 2))])
anomalies = rng.uniform(-3, 7, (20, 2))
Xa = np.vstack([normal, anomalies]); truth = np.r_[np.zeros(980), np.ones(20)]

def precision_at_20(score):          # higher score = more anomalous
    return truth[np.argsort(score)[::-1][:20]].mean()

z = np.abs((Xa - Xa.mean(0)) / Xa.std(0)).max(axis=1)        # largest per-feature z-score
iso = -IsolationForest(n_estimators=300, random_state=0).fit(Xa).score_samples(Xa)
lof = -LocalOutlierFactor(n_neighbors=20).fit(Xa).negative_outlier_factor_
for name, s in [("z-score", z), ("Isolation Forest", iso), ("Local Outlier Factor", lof)]:
    print(f"{name:21}: share of true anomalies among the top 20 {precision_at_20(s):.2f}   ROC-AUC {roc_auc(truth, s):.3f}")

z-score              : share of true anomalies among the top 20 0.50   ROC-AUC 0.730
Isolation Forest     : share of true anomalies among the top 20 0.65   ROC-AUC 0.984
Local Outlier Factor : share of true anomalies among the top 20 0.80   ROC-AUC 0.991


## 8. In Python: a workflow for rare events

In [5]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import precision_recall_curve, precision_score, recall_score, average_precision_score

X, y = make_classification(n_samples=30000, n_features=10, n_informative=5, weights=[0.99],
                           class_sep=1.0, flip_y=0, random_state=0)
X_tr, X_rest, y_tr, y_rest = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
X_val, X_te, y_val, y_te = train_test_split(X_rest, y_rest, test_size=0.5, random_state=0, stratify=y_rest)

model = HistGradientBoostingClassifier(random_state=0).fit(X_tr, y_tr)
p_val, p_te = model.predict_proba(X_val)[:, 1], model.predict_proba(X_te)[:, 1]
print("average precision, validation:", round(average_precision_score(y_val, p_val), 3))

prec, rec, thr = precision_recall_curve(y_val, p_val)
t = thr[np.argmax(prec[:-1] >= 0.8)]                  # lowest threshold with precision of at least 0.8
flag = p_te >= t
print(f"threshold {t:.3f}: test precision {precision_score(y_te, flag):.3f}, recall {recall_score(y_te, flag):.3f}")

average precision, validation: 0.691
threshold 0.458: test precision 0.682, recall 0.500


## 11. Going deeper: correcting probabilities and how isolation works

In [6]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss

X, y = make_classification(n_samples=40000, n_features=10, n_informative=5, weights=[0.98], flip_y=0, random_state=1)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.5, random_state=0, stratify=y)
pi = y_tr.mean()                                                    # true positive rate in training data
p_bal = LogisticRegression(max_iter=1000, class_weight="balanced").fit(X_tr, y_tr).predict_proba(X_te)[:, 1]

# Undo the reweighting: the model behaves as if positives were as common as negatives
odds = p_bal / (1 - p_bal) * (pi / (1 - pi))
p_fixed = odds / (1 + odds)
p_plain = LogisticRegression(max_iter=1000).fit(X_tr, y_tr).predict_proba(X_te)[:, 1]
for name, p in [("balanced weights", p_bal), ("balanced, corrected", p_fixed), ("plain model", p_plain)]:
    print(f"{name:20}: average predicted risk {p.mean():.3f} (true rate {y_te.mean():.3f})   Brier score {brier_score_loss(y_te, p):.4f}")

balanced weights    : average predicted risk 0.372 (true rate 0.020)   Brier score 0.1702
balanced, corrected : average predicted risk 0.017 (true rate 0.020)   Brier score 0.0178
plain model         : average predicted risk 0.020 (true rate 0.020)   Brier score 0.0169


In [7]:

# Isolation Forest's intuition: average number of random splits needed to isolate a point
rng = np.random.default_rng(0)
pts = np.vstack([rng.normal(0, 1, (500, 2)), [[6, 6]]])            # 500 normal points and one far away
def splits_to_isolate(i, trials=200):
    total = 0
    for _ in range(trials):
        idx = np.arange(len(pts)); depth = 0
        while len(idx) > 1:
            f = rng.integers(2); lo, hi = pts[idx, f].min(), pts[idx, f].max()
            t = rng.uniform(lo, hi)
            idx = idx[pts[idx, f] < t] if pts[i, f] < t else idx[pts[idx, f] >= t]
            depth += 1
        total += depth
    return total / trials
print("average splits to isolate the far point:   ", round(splits_to_isolate(500), 1))
print("average splits to isolate a typical point: ", round(np.mean([splits_to_isolate(i, 40) for i in range(10)]), 1))

average splits to isolate the far point:    2.3


average splits to isolate a typical point:  14.6


## Running example: Chennai Homes

The same 12-house dataset used in every lesson (`data/houses.csv` in the course folder).

In [8]:
import pandas as pd
import numpy as np
df = pd.read_csv("../data/houses.csv")
from sklearn.ensemble import IsolationForest
cols = ["size_m2", "price_lakh", "dist_km", "age_yrs"]
iso = IsolationForest(n_estimators=500, random_state=0).fit(df[cols])
df["oddness"] = -iso.score_samples(df[cols])            # higher = more unusual
print(df.sort_values("oddness", ascending=False)[["id", "area", "description", "oddness"]].head(3).round(3).to_string(index=False))

 id      area                 description  oddness
 10 Outskirts  old house needs renovation    0.578
  9 Outskirts    quiet flat far from city    0.558
  8 Outskirts new large villa with garden    0.546


## Coding challenges

Each cell below has **one missing line**, marked `# ← your code here` (or `# ← replace`). Write the line, run the cell, and the check at the bottom tells you whether you got it right. Stuck? A wrong answer prints a hint, and the solutions are at the very end of this notebook.

In [9]:
# Checker used by the challenges below — run this cell first.
import numpy as np
def check(got, want, hint='', atol=1e-6):
    try:
        ok = got is not None and np.shape(got) == np.shape(want) and np.allclose(np.asarray(got, dtype=float), np.asarray(want, dtype=float), rtol=1e-4, atol=atol)
    except Exception:
        ok = False
    print('✅ Correct!' if ok else '❌ Not yet. Hint: ' + hint)


### Challenge 1: Recall and precision from counts

A detector flags 30 cases, 12 of which are real anomalies, out of 20 anomalies in total. Compute recall.

In [10]:
flagged, true_in_flagged, total_anomalies = 30, 12, 20
recall = true_in_flagged / flagged  # ← that is precision; compute recall
check(recall, 0.6, hint='Recall is caught anomalies divided by all anomalies.')

❌ Not yet. Hint: Recall is caught anomalies divided by all anomalies.


### Challenge 2: A z-score detector

Flag the values whose z-score is above 2 in absolute value.

In [11]:
import numpy as np
x = np.array([10, 11, 9, 10, 12, 30, 10, 11])
z = (x - x.mean()) / x.std()
flags = z > 100  # ← replace with the right condition
result = x[flags].tolist()
check(result, [30], hint='Compare the absolute z-score with 2.')

❌ Not yet. Hint: Compare the absolute z-score with 2.


## Your turn

Try the **practice exercises** at the end of the lesson page. Many of them ask you to change one of the cells above and re-run it.

## Challenge solutions

*Spoilers — try the challenges first!*

**1. Recall and precision from counts** — the missing line:

```python
recall = true_in_flagged / total_anomalies
```

**2. A z-score detector** — the missing line:

```python
flags = np.abs(z) > 2
```